In [1]:
import numpy as np
from sklearn.externals.array_api_extra.testing import override

from model_wrapper import *
import cuml.accel
from math import floor
cuml.accel.install()

# of Training Instances: 47
# of Testing Instances: 11
Current RAM usage: 297.31 MB


In [2]:
from sklearn.decomposition import IncrementalPCA

n_components = 20
def data_pipeline(plane):
    _x_train, _y_train = get_data(train_full, "train_series", plane)
    _x_test, _y_test = get_data(test_full, "train_series", plane)

    _x_train = np.reshape(_x_train, shape=(_x_train.shape[0], _x_train.shape[1] * _x_train.shape[2] *  _x_train.shape[3]))
    _x_test = np.reshape(_x_test, shape=(_x_test.shape[0], _x_test.shape[1] * _x_test.shape[2] *  _x_test.shape[3]))

    n_batches = floor(_x_train.shape[0] / n_components)
    inc_pca = IncrementalPCA(n_components=n_components)

    for X_batch in np.array_split(_x_train, n_batches):
        inc_pca.partial_fit(X_batch)

    _x_train = inc_pca.transform(_x_train)
    _x_test = inc_pca.transform(_x_test)
    
    return _x_train, _y_train, _x_test, _y_test

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

for p in planes:
    x_train, y_train, x_test, y_test = data_pipeline(p)

    model = OneVsRestClassifier(LogisticRegression(max_iter=1500))
    model.fit(x_train, y_train)
    pred = model.predict(x_test)

    scores = roc_auc_score(y_test, pred, average=None)
    print(f"{p} Model Score: {np.mean(scores)}")
    for i in range(len(target_columns)):
        print(f"\t{target_columns[i]}: {scores[i]}")
    print("--------")


Sagittal Model Score: 0.530965966903467
	ACL: 0.40259740259740256
	MCL: 0.5
	Medial Meniscus: 0.5238095238095238
	Lateral Meniscus: 0.4666666666666666
	Medial OA: 0.7250000000000001
	Lateral OA: 0.625
	PF OA: 0.5666666666666667
	Effusion: 0.41666666666666663
	Synovitis: 0.6025641025641025
	Baker's: 0.4285714285714286
	Contusion: 0.4230769230769231
	Fracture: 0.6909722222222222
--------
[2026-09-04 08:06:52.111] [CUML] [warning] L-BFGS: max iterations reached
[2026-09-04 08:06:52.112] [CUML] [warning] Maximum iterations reached before solver is converged. To increase model accuracy you can increase the number of iterations (max_iter) or improve the scaling of the input data.
Axial Model Score: 0.5307644901394902
	ACL: 0.6444444444444444
	MCL: 0.4583333333333333
	Medial Meniscus: 0.5833333333333334
	Lateral Meniscus: 0.6041666666666666
	Medial OA: 0.6515151515151515
	Lateral OA: 0.5833333333333334
	PF OA: 0.43333333333333335
	Effusion: 0.35714285714285715
	Synovitis: 0.42857142857142855


In [4]:
class LogRegModel(Model):
    def __init__(self, anatomical_plane, fluid_sensitive=None, fat_suppression=None, n_comp=20):
        self.n_comp = n_comp
        self.inc_pca = IncrementalPCA(n_components=n_comp)
        self.model = OneVsRestClassifier(LogisticRegression(max_iter=1000))
        super().__init__(anatomical_plane, fluid_sensitive, fat_suppression)

    @override
    def train(self, x: np.ndarray, y: np.ndarray):
        print(f"""Training: LogRegModel
\tPlane: {self.anatomical_plane}
\tFluid Sensitive: {self.fluid_sensitive}
\tFat Suppression: {self.fat_suppression}
\tShape: {x.shape}""")
        x = np.reshape(x, shape=(x.shape[0], x.shape[1] * x.shape[2] * x.shape[3]))
        n_batches = floor(x.shape[0] / self.n_comp)

        for X_batch in np.array_split(x, n_batches):
            self.inc_pca.partial_fit(X_batch)

        x_reduced = self.inc_pca.transform(x)
        self.model.fit(x_reduced, y)

    @override
    def predict_batch(self, x: np.ndarray) -> np.ndarray:
        x = np.reshape(x, shape=(x.shape[0], x.shape[1] * x.shape[2] * x.shape[3]))
        x_reduced = self.inc_pca.transform(x)
        return self.model.predict(x_reduced)

    @override
    def predict_instance(self, x: np.ndarray) -> np.ndarray:
        x = np.reshape(x, shape=(1, x.shape[0] * x.shape[1] * x.shape[2]))
        x_reduced = self.inc_pca.transform(x)
        pred_ = self.model.predict(x_reduced)
        return np.reshape(pred_, shape=(pred_.shape[1]))


In [5]:
# Depth 1 ensemble
ensemble = [LogRegModel(p) for p in planes]
scores = Model.get_ensemble_auc_score(ensemble, 1)

print(f"Model Score: {np.mean(scores)}")
for i in range(len(target_columns)):
    print(f"\t{target_columns[i]}: {scores[i]}")

Training: LogRegModel
	Plane: Sagittal
	Fluid Sensitive: None
	Fat Suppression: None
	Shape: (104, 18, 512, 512)
Training: LogRegModel
	Plane: Axial
	Fluid Sensitive: None
	Fat Suppression: None
	Shape: (62, 18, 512, 512)
[2026-09-04 08:10:11.185] [CUML] [warning] L-BFGS: max iterations reached
[2026-09-04 08:10:11.186] [CUML] [warning] Maximum iterations reached before solver is converged. To increase model accuracy you can increase the number of iterations (max_iter) or improve the scaling of the input data.
Training: LogRegModel
	Plane: Coronal
	Fluid Sensitive: None
	Fat Suppression: None
	Shape: (92, 18, 512, 512)
Model Score: 0.529728835978836
	ACL: 0.43333333333333335
	MCL: 0.4444444444444444
	Medial Meniscus: 0.5892857142857142
	Lateral Meniscus: 0.6333333333333333
	Medial OA: 0.7916666666666666
	Lateral OA: 0.6111111111111112
	PF OA: 0.5535714285714286
	Effusion: 0.07142857142857142
	Synovitis: 0.7166666666666667
	Baker's: 0.33333333333333337
	Contusion: 0.25
	Fracture: 0.9285

In [6]:
# Depth 2 ensemble
# fluid sensitive and fat suppression can either be 0 or 1
ensemble = []

for p in planes:
    for i in range(2):
        ensemble.append(LogRegModel(p, i, i, n_comp=7))

scores = Model.get_ensemble_auc_score(ensemble, 2)

print(f"Model Score: {np.mean(scores)}")
for i in range(len(target_columns)):
    print(f"\t{target_columns[i]}: {scores[i]}")

Training: LogRegModel
	Plane: Sagittal
	Fluid Sensitive: 0
	Fat Suppression: 0
	Shape: (56, 18, 512, 512)
Training: LogRegModel
	Plane: Sagittal
	Fluid Sensitive: 1
	Fat Suppression: 1
	Shape: (48, 18, 512, 512)
Training: LogRegModel
	Plane: Axial
	Fluid Sensitive: 0
	Fat Suppression: 0
	Shape: (15, 18, 512, 512)


/home/zero/venvs/ds-venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:469: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/zero/venvs/ds-venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:469: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/zero/venvs/ds-venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:469: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/zero/venvs/ds-venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:469: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Training: LogRegModel
	Plane: Axial
	Fluid Sensitive: 1
	Fat Suppression: 1
	Shape: (47, 18, 512, 512)
Training: LogRegModel
	Plane: Coronal
	Fluid Sensitive: 0
	Fat Suppression: 0
	Shape: (43, 18, 512, 512)
Training: LogRegModel
	Plane: Coronal
	Fluid Sensitive: 1
	Fat Suppression: 1
	Shape: (49, 18, 512, 512)


ValueError: Input contains NaN.

In [ ]:
full_df

In [ ]:
test_series_df